# ResNet vs Plain — CIFAR-10 (akamaster 기반)
런타임 → 런타임 유형 변경 → **GPU(T4)** 선택 후 위에서부터 실행.

- 결과(로그·체크포인트)는 **Google Drive** `MyDrive/resnet_cifar/runs/`에 저장된다.
- 연결이 끊기면 **1~3번 셀을 다시 실행하고 5번 셀을 다시 실행**하면 마지막 epoch부터 이어서 학습한다.
- 학습 설정: 논문 4.2절과 같은 64k iteration (50k장 기준 약 164 epoch), lr 0.1 → 32k, 48k에서 1/10.

In [ ]:
# 1. GPU 확인 + Drive 연결
!nvidia-smi --query-gpu=name,memory.total --format=csv
from google.colab import drive
drive.mount('/content/drive')
RUNS = '/content/drive/MyDrive/resnet_cifar/runs'
FIGS = '/content/drive/MyDrive/resnet_cifar/figures'
import os; os.makedirs(RUNS, exist_ok=True); os.makedirs(FIGS, exist_ok=True)

In [ ]:
# 2. 우리 저장소 받기 (REPO는 우리 GitHub 주소로 바꾸기)
REPO = 'https://github.com/cityhunter0831/pytorch_resnet_cifar10.git'
%cd /content
!rm -rf repo && git clone -q $REPO repo
%cd /content/repo
!git log --oneline -5

In [ ]:
# 3. CIFAR-10을 Colab 로컬 디스크로 다운로드 (Drive보다 훨씬 빠름)
import torchvision
torchvision.datasets.CIFAR10('/content/data', train=True, download=True)
torchvision.datasets.CIFAR10('/content/data', train=False, download=True)

In [ ]:
# 4. 환경 검증: akamaster 사전학습 가중치로 README 수치 재현 (학습 없음, 1분 이내)
!python eval_pretrained.py --data-dir /content/data --archs resnet20 resnet56 --out $FIGS/pretrained_eval.csv

In [ ]:
# 5. 본 실험: 4개 모델을 PARALLEL개씩 동시에 백그라운드로 학습 (이미 끝난 건 자동 skip, 중간이면 이어서)
import subprocess, time, sys
CONFIGS = [('resnet20', False), ('resnet20', True), ('resnet56', False), ('resnet56', True)]  # (arch, plain)
PARALLEL = 2          # T4 + CPU 2개 기준. 첫 epoch 속도 보고 1~3 사이로 조정
ITERS = 64000         # 논문 값. 시간이 정말 부족할 때만 줄이기 (lr 감소 시점은 자동으로 50%/75%)
SEED = 0

def launch(arch, plain):
    name = f"{arch}_{'plain' if plain else 'residual'}_s{SEED}"
    cmd = [sys.executable, 'trainer.py', '-a', arch, '--iters', str(ITERS), '--seed', str(SEED),
           '--data-dir', '/content/data', '--save-dir', RUNS, '--workers', '2'] + (['--plain'] if plain else [])
    log = open(f'{RUNS}/{name}.stdout.txt', 'a')
    return name, subprocess.Popen(cmd, stdout=log, stderr=subprocess.STDOUT)

queue, running = list(CONFIGS), []
while queue or running:
    while queue and len(running) < PARALLEL:
        running.append(launch(*queue.pop(0)))
    time.sleep(60)
    for name, p in list(running):
        last = open(f'{RUNS}/{name}.stdout.txt').read().strip().splitlines()[-1:]
        print(time.strftime('%H:%M'), name, '|', last[0] if last else '')
        if p.poll() is not None:
            running.remove((name, p))
print('ALL DONE')

In [ ]:
# 6. 분석 + 그래프 (학습 중간에 실행해도 됨: 지금까지 결과로 그림)
!python analyze.py --runs $RUNS --out $FIGS
from IPython.display import Image, display
display(Image(f'{FIGS}/fig6_ours.png')); display(Image(f'{FIGS}/grad_by_layer.png'))
import pandas as pd; pd.read_csv(f'{FIGS}/results_table.csv')

In [ ]:
# 7. (선택) 로그 CSV만 묶어서 다운로드 -> 저장소 results/ 에 올리기
!cd /content/drive/MyDrive/resnet_cifar && zip -qr /content/results_csv.zip runs -i '*.csv' '*.json' '*DONE' figures
from google.colab import files; files.download('/content/results_csv.zip')